# 🌪️ Clasificación de Tweets de Desastres con DistilBERT

**Competencia de Kaggle:** [Natural Language Processing with Disaster Tweets](https://www.kaggle.com/c/nlp-getting-started)

**Objetivo:** Predecir si un tweet hace referencia a un desastre real (`target=1`) o no (`target=0`).

**Modelo:** DistilBERT (`distilbert-base-uncased`) con fine-tuning para clasificación binaria.

---

## Flujo del Notebook

1. ⚙️ Configuración inicial e instalación de librerías
2. 📊 Carga y exploración de datos
3. 🔧 Preprocesamiento y tokenización
4. 🤖 Fine-tuning de DistilBERT
5. 📈 Evaluación del modelo
6. 📤 Predicción y generación de envío
7. 💾 Exportación del modelo

---

> **Nota:** Este notebook está optimizado para ejecutarse en un kernel de Kaggle con GPU activada (P100 o T4).
> Asegúrate de habilitar la GPU en *Settings → Accelerator → GPU*.

---
## ⚙️ Paso 1: Configuración Inicial

Instalamos las librerías necesarias y configuramos el entorno. Usamos **PyTorch** como backend junto con la librería `transformers` de HuggingFace, que proporciona acceso directo a DistilBERT con soporte nativo para fine-tuning.

**¿Por qué transformers y no keras_nlp?**
- `transformers` de HuggingFace tiene mayor compatibilidad y soporte para DistilBERT.
- Facilita el uso de `Trainer` API para fine-tuning con pocas líneas de código.
- Mayor comunidad y documentación disponible.

In [ ]:
# ============================================================
# Instalación de librerías necesarias
# En Kaggle, muchas ya están preinstaladas; instalamos las faltantes
# ============================================================
import subprocess, sys

paquetes = [
    "transformers==4.40.0",
    "datasets==2.19.0",
    "accelerate==0.29.3",
    "scikit-learn",
    "tqdm",
]

for paquete in paquetes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

print("✅ Instalación completada")

In [ ]:
# ============================================================
# Importaciones principales
# ============================================================
import os
import re
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# HuggingFace Transformers
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    accuracy_score,
)

print(f"✅ Importaciones completadas")
print(f"   PyTorch versión: {torch.__version__}")

In [ ]:
# ============================================================
# Configuración de la GPU y semilla aleatoria
# ============================================================

# Reproducibilidad: fijamos semillas en todas las librerías relevantes
SEMILLA = 42

def fijar_semilla(semilla: int = SEMILLA):
    """Fija semillas para reproducibilidad completa del experimento."""
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)
    # Hace que cuDNN sea determinístico (puede ralentizar un poco)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semilla(SEMILLA)

# Detectar dispositivo disponible
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo activo: {DISPOSITIVO}")

if torch.cuda.is_available():
    nombre_gpu = torch.cuda.get_device_name(0)
    memoria_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU detectada: {nombre_gpu}")
    print(f"   Memoria GPU total: {memoria_gb:.1f} GB")
else:
    print("⚠️  GPU no disponible. El entrenamiento será más lento en CPU.")
    print("   Activa la GPU en: Settings → Accelerator → GPU")

---
## 📊 Paso 2: Carga y Exploración de Datos

Cargamos los tres archivos del dataset:
- `train.csv`: datos etiquetados para entrenamiento.
- `test.csv`: datos sin etiquetar para predicción final.
- `sample_submission.csv`: formato esperado del envío.

**Columnas del dataset:**
| Columna | Descripción |
|---------|-------------|
| `id` | Identificador único del tweet |
| `keyword` | Palabra clave asociada (puede ser nula) |
| `location` | Ubicación del usuario (puede ser nula) |
| `text` | Texto del tweet |
| `target` | 1 = desastre real, 0 = no desastre |


In [ ]:
# ============================================================
# Carga de datos desde la ruta estándar de Kaggle
# ============================================================
RUTA_BASE = "/kaggle/input/nlp-getting-started/"

df_train = pd.read_csv(os.path.join(RUTA_BASE, "train.csv"))
df_test  = pd.read_csv(os.path.join(RUTA_BASE, "test.csv"))
df_sub   = pd.read_csv(os.path.join(RUTA_BASE, "sample_submission.csv"))

print(f"📂 Conjunto de entrenamiento: {df_train.shape[0]:,} filas × {df_train.shape[1]} columnas")
print(f"📂 Conjunto de prueba:        {df_test.shape[0]:,} filas × {df_test.shape[1]} columnas")
print(f"📂 Envío de muestra:          {df_sub.shape[0]:,} filas × {df_sub.shape[1]} columnas")
print()
print("── Primeras filas del conjunto de entrenamiento ──")
df_train.head()

In [ ]:
# ============================================================
# Análisis Exploratorio de Datos (EDA)
# ============================================================

# --- Distribución de clases ---
conteo_clases = df_train["target"].value_counts()
porcentajes   = df_train["target"].value_counts(normalize=True) * 100

print("📊 Distribución de clases:")
print(f"   No desastre (0): {conteo_clases[0]:,} tweets ({porcentajes[0]:.1f}%)")
print(f"   Desastre    (1): {conteo_clases[1]:,} tweets ({porcentajes[1]:.1f}%)")
print()

# --- Valores nulos ---
print("🔍 Valores nulos por columna (entrenamiento):")
print(df_train.isnull().sum())
print()

# --- Longitud de tweets ---
df_train["longitud"] = df_train["text"].str.len()
df_test["longitud"]  = df_test["text"].str.len()

print("📏 Estadísticas de longitud de tweets (entrenamiento):")
print(df_train["longitud"].describe().round(1))

In [ ]:
# ============================================================
# Visualizaciones del EDA
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Análisis Exploratorio de Datos", fontsize=15, fontweight="bold")

# Gráfico 1: Distribución de clases
colores = ["#4ECDC4", "#FF6B6B"]
axes[0].bar(["No Desastre (0)", "Desastre (1)"], conteo_clases.values, color=colores, edgecolor="black")
axes[0].set_title("Distribución de Clases", fontweight="bold")
axes[0].set_ylabel("Número de Tweets")
for i, v in enumerate(conteo_clases.values):
    axes[0].text(i, v + 20, f"{v:,}\n({porcentajes.values[i]:.1f}%)",
                 ha="center", fontweight="bold")

# Gráfico 2: Distribución de longitud por clase
for etiqueta, color in zip([0, 1], colores):
    subset = df_train[df_train["target"] == etiqueta]["longitud"]
    nombre = "No Desastre" if etiqueta == 0 else "Desastre"
    axes[1].hist(subset, bins=40, alpha=0.6, color=color, label=nombre, edgecolor="white")
axes[1].set_title("Longitud de Tweets por Clase", fontweight="bold")
axes[1].set_xlabel("Número de Caracteres")
axes[1].set_ylabel("Frecuencia")
axes[1].legend()

# Gráfico 3: Palabras clave más frecuentes
top_keywords = (df_train["keyword"]
                .dropna()
                .value_counts()
                .head(15))
axes[2].barh(top_keywords.index[::-1], top_keywords.values[::-1], color="#A8DADC", edgecolor="black")
axes[2].set_title("Top 15 Palabras Clave", fontweight="bold")
axes[2].set_xlabel("Frecuencia")

plt.tight_layout()
plt.savefig("/kaggle/working/eda_visualizaciones.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Gráficos guardados en /kaggle/working/eda_visualizaciones.png")

In [ ]:
# ============================================================
# Ejemplos de tweets por clase
# ============================================================
print("📝 Ejemplos de tweets de DESASTRE REAL (target=1):")
print("-" * 60)
for tweet in df_train[df_train["target"] == 1]["text"].sample(3, random_state=SEMILLA):
    print(f"  • {tweet}")
print()
print("📝 Ejemplos de tweets que NO son desastre (target=0):")
print("-" * 60)
for tweet in df_train[df_train["target"] == 0]["text"].sample(3, random_state=SEMILLA):
    print(f"  • {tweet}")

---
## 🔧 Paso 3: Preprocesamiento y Tokenización

### Decisiones de preprocesamiento:

| Transformación | ¿Aplicar? | Justificación |
|---|---|---|
| Eliminar URLs | ✅ Sí | Las URLs no aportan semántica útil al modelo |
| Eliminar menciones (@usuario) | ✅ Sí | Son ruido; no aportan contexto semántico |
| Eliminar caracteres especiales | ⚠️ Parcial | Conservamos signos de puntuación; DistilBERT puede procesarlos |
| Eliminar emojis | ❌ No | Los emojis pueden indicar tono (alarma, urgencia) útil para el modelo |
| Convertir a minúsculas | ❌ No | El tokenizador `uncased` ya lo hace internamente |
| Stemming/Lemmatización | ❌ No | BERT-based no necesita reducción de palabras |

### Tokenización:
Usamos `DistilBertTokenizerFast` con:
- **max_length = 128**: cubre el 95%+ de los tweets (máx. ~140 chars originales).
- **padding = 'max_length'**: rellena secuencias cortas.
- **truncation = True**: trunca secuencias largas.

In [ ]:
# ============================================================
# Función de limpieza de texto
# ============================================================

def limpiar_tweet(texto: str) -> str:
    """
    Limpia el texto de un tweet eliminando ruido que no aporta
    información semántica al modelo de clasificación.
    
    Pasos:
      1. Eliminar URLs (http/https/www)
      2. Eliminar menciones de usuario (@usuario)
      3. Eliminar caracteres repetidos (ej: 'aaaaaaa' → 'aa')
      4. Eliminar espacios múltiples
      5. Strip de espacios en extremos
    """
    # 1. Eliminar URLs
    texto = re.sub(r"http\S+|www\.\S+", "", texto)
    
    # 2. Eliminar menciones de usuario
    texto = re.sub(r"@\w+", "", texto)
    
    # 3. Reducir caracteres repetidos (más de 2 veces) a 2 repeticiones
    # Ej: 'nooooo' → 'noo' (conserva la intención expresiva)
    texto = re.sub(r"(.)\1{2,}", r"\1\1", texto)
    
    # 4. Eliminar espacios múltiples
    texto = re.sub(r"\s+", " ", texto)
    
    # 5. Limpiar extremos
    return texto.strip()


# Aplicar limpieza a entrenamiento y prueba
df_train["texto_limpio"] = df_train["text"].apply(limpiar_tweet)
df_test["texto_limpio"]  = df_test["text"].apply(limpiar_tweet)

# Verificar el efecto de la limpieza con un ejemplo
idx_ejemplo = df_train[df_train["text"].str.contains("http", na=False)].index[0]
print("🔍 Ejemplo de limpieza:")
print(f"   Original:  {df_train.loc[idx_ejemplo, 'text']}")
print(f"   Limpio:    {df_train.loc[idx_ejemplo, 'texto_limpio']}")
print()
print(f"✅ Preprocesamiento aplicado a {len(df_train):,} tweets de entrenamiento")
print(f"✅ Preprocesamiento aplicado a {len(df_test):,} tweets de prueba")

In [ ]:
# ============================================================
# Inicialización del tokenizador de DistilBERT
# ============================================================

NOMBRE_MODELO  = "distilbert-base-uncased"  # Modelo preentrenado a usar
MAX_LONGITUD   = 128                         # Longitud máxima de tokens

# Cargar el tokenizador "rápido" (implementación en Rust, mucho más veloz)
tokenizador = DistilBertTokenizerFast.from_pretrained(NOMBRE_MODELO)

print(f"✅ Tokenizador cargado: {NOMBRE_MODELO}")
print(f"   Tamaño del vocabulario: {tokenizador.vocab_size:,} tokens")
print(f"   Longitud máxima configurada: {MAX_LONGITUD} tokens")
print()

# Demostración de tokenización
tweet_ejemplo = df_train["texto_limpio"].iloc[0]
tokens_ejemplo = tokenizador.tokenize(tweet_ejemplo)
print(f"📝 Tweet de ejemplo: '{tweet_ejemplo[:80]}...'")
print(f"   Tokens: {tokens_ejemplo[:15]}...")
print(f"   Total de tokens: {len(tokens_ejemplo)}")

In [ ]:
# ============================================================
# Dataset personalizado de PyTorch
# ============================================================

class TweetDataset(Dataset):
    """
    Dataset de PyTorch para tweets.
    
    Tokeniza los textos en el momento de carga y los convierte
    a tensores para ser usados directamente por el modelo.
    """
    
    def __init__(self, textos, etiquetas=None, tokenizador=None, max_longitud=128):
        """
        Args:
            textos (list): Lista de strings con los tweets.
            etiquetas (list, opcional): Lista de etiquetas (0 o 1).
            tokenizador: Tokenizador de HuggingFace.
            max_longitud (int): Longitud máxima de la secuencia tokenizada.
        """
        self.textos       = textos
        self.etiquetas    = etiquetas
        self.tokenizador  = tokenizador
        self.max_longitud = max_longitud
    
    def __len__(self):
        return len(self.textos)
    
    def __getitem__(self, idx):
        # Tokenizar el texto
        codificacion = self.tokenizador(
            self.textos[idx],
            max_length=self.max_longitud,
            padding="max_length",    # Rellena hasta max_length
            truncation=True,          # Trunca si excede max_length
            return_tensors="pt",      # Retorna tensores de PyTorch
        )
        
        # Construir el ítem con los tensores relevantes
        item = {
            "input_ids":      codificacion["input_ids"].squeeze(0),
            "attention_mask": codificacion["attention_mask"].squeeze(0),
        }
        
        # Si hay etiquetas (entrenamiento/validación), incluirlas
        if self.etiquetas is not None:
            item["labels"] = torch.tensor(self.etiquetas[idx], dtype=torch.long)
        
        return item


print("✅ Clase TweetDataset definida correctamente")
print(f"   Tokenización: padding='max_length', truncation=True, max_length={MAX_LONGITUD}")

---
## 🤖 Paso 4: Fine-tuning de DistilBERT

### Arquitectura del modelo:
```
DistilBERT (6 capas Transformer)
    ↓
Token [CLS] → representación del tweet
    ↓
Capa Dropout (p=0.3)
    ↓
Capa Lineal (768 → 2)
    ↓
Softmax → [P(no desastre), P(desastre)]
```

### Hiperparámetros:
| Parámetro | Valor | Justificación |
|---|---|---|
| Learning Rate | 2e-5 | Estándar para fine-tuning BERT |
| Épocas | 4 | Suficiente para convergencia sin sobreajuste |
| Batch Size | 32 | Balance entre velocidad y estabilidad |
| Warmup Steps | 10% | Previene destrucción de pesos preentrenados |
| Weight Decay | 0.01 | Regularización L2 ligera |

In [ ]:
# ============================================================
# Hiperparámetros del entrenamiento
# ============================================================

CONFIG = {
    "modelo_base":       "distilbert-base-uncased",
    "max_longitud":       128,
    "batch_size":         32,
    "epocas":             4,
    "learning_rate":      2e-5,
    "weight_decay":       0.01,
    "warmup_ratio":       0.1,    # 10% de los pasos para warmup
    "proporcion_val":     0.2,    # 20% para validación
    "num_clases":         2,
    "semilla":            SEMILLA,
}

print("⚙️  Configuración del experimento:")
for clave, valor in CONFIG.items():
    print(f"   {clave:<20}: {valor}")

In [ ]:
# ============================================================
# División estratificada en entrenamiento y validación
# Estratificada = mantiene la proporción de clases en ambos sets
# ============================================================

X = df_train["texto_limpio"].tolist()
y = df_train["target"].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=CONFIG["proporcion_val"],
    random_state=CONFIG["semilla"],
    stratify=y,           # División estratificada
)

print(f"✅ División completada:")
print(f"   Entrenamiento: {len(X_train):,} tweets")
print(f"   Validación:    {len(X_val):,} tweets")
print()
print("   Distribución de clases en entrenamiento:")
for clase, count in zip(*np.unique(y_train, return_counts=True)):
    print(f"     Clase {clase}: {count:,} ({count/len(y_train)*100:.1f}%)")
print("   Distribución de clases en validación:")
for clase, count in zip(*np.unique(y_val, return_counts=True)):
    print(f"     Clase {clase}: {count:,} ({count/len(y_val)*100:.1f}%)")

In [ ]:
# ============================================================
# Crear Datasets y DataLoaders de PyTorch
# ============================================================

# Datasets
dataset_train = TweetDataset(X_train, y_train, tokenizador, CONFIG["max_longitud"])
dataset_val   = TweetDataset(X_val,   y_val,   tokenizador, CONFIG["max_longitud"])
dataset_test  = TweetDataset(
    df_test["texto_limpio"].tolist(),
    etiquetas=None,    # Sin etiquetas en test
    tokenizador=tokenizador,
    max_longitud=CONFIG["max_longitud"]
)

# DataLoaders: iteradores en batches para el entrenamiento
loader_train = DataLoader(
    dataset_train,
    batch_size=CONFIG["batch_size"],
    shuffle=True,      # Mezclar en cada época (entrenamiento)
    num_workers=2,     # Carga en paralelo
    pin_memory=True,   # Optimización para transferencia CPU→GPU
)

loader_val = DataLoader(
    dataset_val,
    batch_size=CONFIG["batch_size"] * 2,  # Batch más grande en eval (no hay gradientes)
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

loader_test = DataLoader(
    dataset_test,
    batch_size=CONFIG["batch_size"] * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"✅ DataLoaders creados:")
print(f"   Batches de entrenamiento: {len(loader_train):,}")
print(f"   Batches de validación:    {len(loader_val):,}")
print(f"   Batches de prueba:        {len(loader_test):,}")

In [ ]:
# ============================================================
# Cargar el modelo DistilBERT preentrenado con cabeza de clasificación
# ============================================================
# DistilBertForSequenceClassification ya incluye:
#   - DistilBERT base (6 capas Transformer, 66M parámetros)
#   - Pre-clasificador: Linear(768, 768) + ReLU + Dropout
#   - Clasificador final: Linear(768, num_labels)

modelo = DistilBertForSequenceClassification.from_pretrained(
    CONFIG["modelo_base"],
    num_labels=CONFIG["num_clases"],
)

# Mover el modelo al dispositivo (GPU/CPU)
modelo = modelo.to(DISPOSITIVO)

# Contar parámetros totales y entrenables
total_params     = sum(p.numel() for p in modelo.parameters())
params_entrena   = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

print(f"✅ Modelo cargado: {CONFIG['modelo_base']}")
print(f"   Parámetros totales:     {total_params:,}")
print(f"   Parámetros entrenables: {params_entrena:,}")
print(f"   Dispositivo:            {next(modelo.parameters()).device}")

In [ ]:
# ============================================================
# Optimizador AdamW y Scheduler de Learning Rate
# ============================================================

# Número total de pasos de actualización de pesos
pasos_totales  = len(loader_train) * CONFIG["epocas"]
pasos_warmup   = int(pasos_totales * CONFIG["warmup_ratio"])

# AdamW: Adam con weight decay desacoplado (recomendado para BERT)
# Excluimos bias y LayerNorm del weight decay (práctica estándar)
sin_decay = ["bias", "LayerNorm.weight"]
grupos_params = [
    {
        "params": [p for n, p in modelo.named_parameters() if not any(nd in n for nd in sin_decay)],
        "weight_decay": CONFIG["weight_decay"],
    },
    {
        "params": [p for n, p in modelo.named_parameters() if any(nd in n for nd in sin_decay)],
        "weight_decay": 0.0,
    },
]

optimizador = AdamW(grupos_params, lr=CONFIG["learning_rate"])

# Scheduler: warmup lineal + decaimiento lineal
# Durante warmup, el LR sube de 0 → lr_max
# Luego decrece linealmente hasta 0
scheduler = get_linear_schedule_with_warmup(
    optimizador,
    num_warmup_steps=pasos_warmup,
    num_training_steps=pasos_totales,
)

print(f"⚙️  Optimizador y scheduler configurados:")
print(f"   Pasos totales de entrenamiento: {pasos_totales:,}")
print(f"   Pasos de warmup:                {pasos_warmup:,} ({CONFIG['warmup_ratio']*100:.0f}%)")
print(f"   Learning rate inicial:          {CONFIG['learning_rate']}")

In [ ]:
# ============================================================
# Funciones de entrenamiento y evaluación por época
# ============================================================

def entrenar_una_epoca(modelo, loader, optimizador, scheduler, dispositivo):
    """
    Ejecuta una época completa de entrenamiento.
    
    Returns:
        tuple: (pérdida_promedio, f1_score)
    """
    modelo.train()  # Modo entrenamiento (activa Dropout, BatchNorm, etc.)
    perdida_total = 0
    todas_preds   = []
    todas_labels  = []
    
    barra = tqdm(loader, desc="  Entrenando", leave=False)
    
    for batch in barra:
        # Mover tensores al dispositivo
        input_ids      = batch["input_ids"].to(dispositivo)
        attention_mask = batch["attention_mask"].to(dispositivo)
        labels         = batch["labels"].to(dispositivo)
        
        # Forward pass
        optimizador.zero_grad()   # Limpiar gradientes acumulados
        salidas = modelo(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        
        perdida = salidas.loss
        logits  = salidas.logits
        
        # Backward pass
        perdida.backward()                            # Calcular gradientes
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)  # Recortar gradientes
        optimizador.step()                            # Actualizar pesos
        scheduler.step()                              # Actualizar learning rate
        
        # Acumular métricas
        perdida_total += perdida.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        todas_preds.extend(preds)
        todas_labels.extend(labels.cpu().numpy())
        
        # Actualizar barra de progreso
        barra.set_postfix({"loss": f"{perdida.item():.4f}"})
    
    perdida_prom = perdida_total / len(loader)
    f1 = f1_score(todas_labels, todas_preds, average="binary")
    return perdida_prom, f1


def evaluar(modelo, loader, dispositivo):
    """
    Evalúa el modelo en un conjunto de datos.
    
    Returns:
        tuple: (pérdida_promedio, f1_score, predicciones, probabilidades, etiquetas_verdaderas)
    """
    modelo.eval()  # Modo evaluación (desactiva Dropout)
    perdida_total  = 0
    todas_preds    = []
    todas_probs    = []
    todas_labels   = []
    
    with torch.no_grad():  # Sin cálculo de gradientes (ahorra memoria y tiempo)
        for batch in tqdm(loader, desc="  Evaluando", leave=False):
            input_ids      = batch["input_ids"].to(dispositivo)
            attention_mask = batch["attention_mask"].to(dispositivo)
            labels         = batch["labels"].to(dispositivo)
            
            salidas = modelo(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            
            perdida_total += salidas.loss.item()
            
            # Probabilidades con Softmax
            probs = torch.softmax(salidas.logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(salidas.logits, dim=1).cpu().numpy()
            
            todas_probs.extend(probs)
            todas_preds.extend(preds)
            todas_labels.extend(labels.cpu().numpy())
    
    perdida_prom = perdida_total / len(loader)
    f1 = f1_score(todas_labels, todas_preds, average="binary")
    
    return perdida_prom, f1, np.array(todas_preds), np.array(todas_probs), np.array(todas_labels)


print("✅ Funciones de entrenamiento y evaluación definidas")

In [ ]:
# ============================================================
# Bucle de entrenamiento principal con Early Stopping
# ============================================================

historial = {
    "perdida_train": [], "f1_train": [],
    "perdida_val":   [], "f1_val":   [],
}

# Early Stopping: detiene si no mejora después de N épocas
PACIENCIA          = 2
mejor_f1_val       = 0.0
contador_paciencia = 0
RUTA_MEJOR_MODELO  = "/kaggle/working/mejor_modelo.pt"

print("🚀 Iniciando entrenamiento...")
print(f"   Épocas: {CONFIG['epocas']} | Batch size: {CONFIG['batch_size']} | LR: {CONFIG['learning_rate']}")
print("=" * 65)

for epoca in range(1, CONFIG["epocas"] + 1):
    print(f"\n📅 Época {epoca}/{CONFIG['epocas']}")
    
    # --- Entrenamiento ---
    perdida_tr, f1_tr = entrenar_una_epoca(
        modelo, loader_train, optimizador, scheduler, DISPOSITIVO
    )
    
    # --- Validación ---
    perdida_vl, f1_vl, preds_vl, probs_vl, labels_vl = evaluar(
        modelo, loader_val, DISPOSITIVO
    )
    
    # Guardar historial
    historial["perdida_train"].append(perdida_tr)
    historial["f1_train"].append(f1_tr)
    historial["perdida_val"].append(perdida_vl)
    historial["f1_val"].append(f1_vl)
    
    lr_actual = scheduler.get_last_lr()[0]
    
    print(f"   Loss  → Train: {perdida_tr:.4f} | Val: {perdida_vl:.4f}")
    print(f"   F1    → Train: {f1_tr:.4f} | Val: {f1_vl:.4f}")
    print(f"   LR actual: {lr_actual:.2e}")
    
    # --- Early Stopping y guardado del mejor modelo ---
    if f1_vl > mejor_f1_val:
        mejor_f1_val = f1_vl
        contador_paciencia = 0
        torch.save(modelo.state_dict(), RUTA_MEJOR_MODELO)
        print(f"   ✅ Nuevo mejor modelo guardado (F1 val: {mejor_f1_val:.4f})")
    else:
        contador_paciencia += 1
        print(f"   ⚠️  Sin mejora. Paciencia: {contador_paciencia}/{PACIENCIA}")
        if contador_paciencia >= PACIENCIA:
            print(f"   🛑 Early stopping activado en época {epoca}")
            break

print(f"\n{'='*65}")
print(f"🏆 Mejor F1 en validación: {mejor_f1_val:.4f}")

In [ ]:
# ============================================================
# Visualización de curvas de entrenamiento
# ============================================================

epocas_rango = range(1, len(historial["perdida_train"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Curvas de Entrenamiento", fontsize=14, fontweight="bold")

# Curva de pérdida
axes[0].plot(epocas_rango, historial["perdida_train"], "o-", label="Entrenamiento", color="#2196F3")
axes[0].plot(epocas_rango, historial["perdida_val"],   "o-", label="Validación",    color="#FF5722")
axes[0].set_title("Pérdida (Cross-Entropy)", fontweight="bold")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Pérdida")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Curva de F1
axes[1].plot(epocas_rango, historial["f1_train"], "o-", label="Entrenamiento", color="#4CAF50")
axes[1].plot(epocas_rango, historial["f1_val"],   "o-", label="Validación",    color="#FF9800")
axes[1].set_title("F1-Score Binario", fontweight="bold")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("F1-Score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig("/kaggle/working/curvas_entrenamiento.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Curvas guardadas en /kaggle/working/curvas_entrenamiento.png")

---
## 📈 Paso 5: Evaluación del Modelo

Cargamos el mejor modelo guardado y lo evaluamos exhaustivamente en el conjunto de validación con:
- **Matriz de Confusión**: Visualiza TP, TN, FP, FN
- **Reporte de Clasificación**: Precisión, Recall, F1 por clase
- **Curva ROC y AUC**: Mide la capacidad discriminativa del modelo
- **Curva Precisión-Recall**: Más informativa que ROC en datasets desbalanceados

In [ ]:
# ============================================================
# Cargar el mejor modelo guardado durante el entrenamiento
# ============================================================

print(f"🔄 Cargando mejor modelo desde: {RUTA_MEJOR_MODELO}")
modelo.load_state_dict(torch.load(RUTA_MEJOR_MODELO, map_location=DISPOSITIVO))
modelo.eval()
print(f"✅ Modelo cargado correctamente")
print(f"   F1 esperado en validación: {mejor_f1_val:.4f}")

In [ ]:
# ============================================================
# Evaluación completa en el conjunto de validación
# ============================================================

perdida_final, f1_final, preds_final, probs_final, labels_final = evaluar(
    modelo, loader_val, DISPOSITIVO
)

# Métricas globales
accuracy  = accuracy_score(labels_final, preds_final)
auc_score = roc_auc_score(labels_final, probs_final)

print("\n" + "="*55)
print("📊 MÉTRICAS FINALES EN VALIDACIÓN")
print("="*55)
print(f"  Accuracy:  {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  F1-Score:  {f1_final:.4f}")
print(f"  AUC-ROC:   {auc_score:.4f}")
print(f"  Loss:      {perdida_final:.4f}")
print("="*55)
print()
print("📋 Reporte de Clasificación detallado:")
print(classification_report(
    labels_final, preds_final,
    target_names=["No Desastre (0)", "Desastre (1)"]
))

In [ ]:
# ============================================================
# Visualizaciones de evaluación
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Evaluación del Modelo en Validación", fontsize=14, fontweight="bold")

# --- 1. Matriz de Confusión ---
mc = confusion_matrix(labels_final, preds_final)
sns.heatmap(
    mc, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No Desastre", "Desastre"],
    yticklabels=["No Desastre", "Desastre"],
    ax=axes[0], linewidths=0.5
)
axes[0].set_title("Matriz de Confusión", fontweight="bold")
axes[0].set_ylabel("Verdadero")
axes[0].set_xlabel("Predicho")

# Añadir anotaciones de TN/FP/FN/TP
tn, fp, fn, tp = mc.ravel()
print(f"\n🔢 Desglose de la Matriz de Confusión:")
print(f"   VP (Verdaderos Positivos):  {tp:,}")
print(f"   VN (Verdaderos Negativos):  {tn:,}")
print(f"   FP (Falsos Positivos):       {fp:,}")
print(f"   FN (Falsos Negativos):       {fn:,}")

# --- 2. Curva ROC ---
fpr, tpr, umbrales = roc_curve(labels_final, probs_final)
axes[1].plot(fpr, tpr, color="#2196F3", lw=2, label=f"AUC = {auc_score:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray", lw=1, label="Aleatorio (AUC=0.5)")
axes[1].fill_between(fpr, tpr, alpha=0.1, color="#2196F3")
axes[1].set_title("Curva ROC", fontweight="bold")
axes[1].set_xlabel("Tasa de Falsos Positivos")
axes[1].set_ylabel("Tasa de Verdaderos Positivos")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

# --- 3. Distribución de probabilidades por clase ---
for clase, color, etiq in zip([0, 1], ["#4ECDC4", "#FF6B6B"], ["No Desastre", "Desastre"]):
    mask = labels_final == clase
    axes[2].hist(probs_final[mask], bins=40, alpha=0.6, color=color, label=etiq, density=True)
axes[2].axvline(0.5, color="black", linestyle="--", label="Umbral (0.5)")
axes[2].set_title("Distribución de Probabilidades", fontweight="bold")
axes[2].set_xlabel("P(Desastre)")
axes[2].set_ylabel("Densidad")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/evaluacion_modelo.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📊 Gráficos guardados en /kaggle/working/evaluacion_modelo.png")

---
## 📤 Paso 6: Predicción y Generación del Archivo de Envío

Usamos el mejor modelo para predecir sobre el conjunto de prueba y generamos `submission.csv`.

**Umbral de decisión:** Usamos 0.5 por defecto. Si el modelo muestra sesgo hacia una clase, se puede ajustar este umbral optimizándolo en validación.

In [ ]:
# ============================================================
# Optimización del umbral de decisión en validación
# ============================================================

# Probamos diferentes umbrales para maximizar el F1 en validación
umbrales_prueba = np.arange(0.3, 0.71, 0.01)
f1_por_umbral   = []

for u in umbrales_prueba:
    preds_u = (probs_final >= u).astype(int)
    f1_u    = f1_score(labels_final, preds_u, average="binary")
    f1_por_umbral.append(f1_u)

# Encontrar el umbral óptimo
idx_optimo  = np.argmax(f1_por_umbral)
umbral_opt  = umbrales_prueba[idx_optimo]
f1_optimo   = f1_por_umbral[idx_optimo]

print(f"🎯 Optimización del umbral de decisión:")
print(f"   Umbral por defecto (0.5):  F1 = {f1_score(labels_final, (probs_final>=0.5).astype(int)):.4f}")
print(f"   Umbral óptimo ({umbral_opt:.2f}):     F1 = {f1_optimo:.4f}")
print()

# Visualizar
plt.figure(figsize=(10, 4))
plt.plot(umbrales_prueba, f1_por_umbral, "b-", linewidth=2)
plt.axvline(umbral_opt, color="red", linestyle="--", label=f"Óptimo: {umbral_opt:.2f} (F1={f1_optimo:.4f})")
plt.axvline(0.5, color="gray", linestyle=":", label="Default: 0.5")
plt.xlabel("Umbral de Decisión")
plt.ylabel("F1-Score")
plt.title("F1-Score vs Umbral de Decisión", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/umbral_optimo.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Predecir sobre el conjunto de prueba
# ============================================================

print("🔮 Generando predicciones sobre el conjunto de prueba...")

modelo.eval()
probs_test = []

with torch.no_grad():
    for batch in tqdm(loader_test, desc="Prediciendo"):
        input_ids      = batch["input_ids"].to(DISPOSITIVO)
        attention_mask = batch["attention_mask"].to(DISPOSITIVO)
        
        salidas = modelo(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        
        # Probabilidad de la clase positiva (desastre=1)
        probs = torch.softmax(salidas.logits, dim=1)[:, 1].cpu().numpy()
        probs_test.extend(probs)

probs_test  = np.array(probs_test)
preds_test  = (probs_test >= umbral_opt).astype(int)

print(f"✅ Predicciones generadas: {len(preds_test):,} tweets")
print(f"   Usando umbral óptimo: {umbral_opt:.2f}")
print()
print("📊 Distribución de predicciones en test:")
clases_u, conteos = np.unique(preds_test, return_counts=True)
for c, n in zip(clases_u, conteos):
    etiq = "No Desastre" if c == 0 else "Desastre"
    print(f"   {etiq} ({c}): {n:,} tweets ({n/len(preds_test)*100:.1f}%)")

In [ ]:
# ============================================================
# Generar el archivo de envío a Kaggle
# ============================================================

RUTA_SUBMISSION = "/kaggle/working/submission.csv"

df_submission = pd.DataFrame({
    "id":     df_test["id"],
    "target": preds_test,
})

df_submission.to_csv(RUTA_SUBMISSION, index=False)

print(f"✅ Archivo de envío generado: {RUTA_SUBMISSION}")
print(f"   Forma: {df_submission.shape}")
print()
print("📋 Primeras filas del envío:")
print(df_submission.head(10).to_string(index=False))
print()

# Validar que el formato sea correcto
assert list(df_submission.columns) == ["id", "target"], "❌ Error: columnas incorrectas"
assert df_submission["target"].isin([0, 1]).all(), "❌ Error: valores de target inválidos"
assert len(df_submission) == len(df_test), "❌ Error: número de filas incorrecto"
print("✅ Validación del archivo de envío: CORRECTO")

---
## 💾 Paso 7: Exportación del Modelo Fine-tuned

Guardamos el modelo completo en el formato nativo de HuggingFace para facilitar su reutilización:
- `config.json`: configuración de la arquitectura
- `pytorch_model.bin` / `model.safetensors`: pesos del modelo
- `tokenizer.json`: tokenizador completo

Con esto se puede cargar el modelo en el futuro con `from_pretrained(ruta_local)`.

In [ ]:
# ============================================================
# Guardar el modelo fine-tuned en formato HuggingFace
# ============================================================

RUTA_MODELO_FINAL = "/kaggle/working/distilbert_tweets_desastre"
os.makedirs(RUTA_MODELO_FINAL, exist_ok=True)

# Guardar pesos del modelo
modelo.save_pretrained(RUTA_MODELO_FINAL)

# Guardar tokenizador
tokenizador.save_pretrained(RUTA_MODELO_FINAL)

# Guardar configuración del experimento
import json
config_guardar = {**CONFIG, "mejor_f1_validacion": float(mejor_f1_val), "umbral_optimo": float(umbral_opt)}
with open(os.path.join(RUTA_MODELO_FINAL, "experimento_config.json"), "w") as f:
    json.dump(config_guardar, f, indent=2, ensure_ascii=False)

print(f"✅ Modelo guardado en: {RUTA_MODELO_FINAL}")
print("   Archivos guardados:")
for archivo in sorted(os.listdir(RUTA_MODELO_FINAL)):
    tamanio = os.path.getsize(os.path.join(RUTA_MODELO_FINAL, archivo)) / 1e6
    print(f"   • {archivo:40s} ({tamanio:.1f} MB)")

In [ ]:
# ============================================================
# Función de inferencia para nuevos tweets (producción)
# ============================================================

def predecir_tweet(texto: str, modelo, tokenizador, dispositivo, umbral: float = 0.5) -> dict:
    """
    Predice si un tweet hace referencia a un desastre real.
    
    Args:
        texto (str): Tweet a clasificar.
        modelo: Modelo fine-tuned.
        tokenizador: Tokenizador de DistilBERT.
        dispositivo: CPU o GPU.
        umbral (float): Umbral de decisión (default: 0.5).
    
    Returns:
        dict: Predicción y probabilidades.
    """
    texto_limpio = limpiar_tweet(texto)
    
    codificacion = tokenizador(
        texto_limpio,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    
    modelo.eval()
    with torch.no_grad():
        salidas = modelo(
            input_ids=codificacion["input_ids"].to(dispositivo),
            attention_mask=codificacion["attention_mask"].to(dispositivo),
        )
    
    probs = torch.softmax(salidas.logits, dim=1).cpu().numpy()[0]
    pred  = int(probs[1] >= umbral)
    
    return {
        "tweet":             texto[:100],
        "prediccion":        "🔴 DESASTRE" if pred == 1 else "🟢 No Desastre",
        "p_no_desastre":     f"{probs[0]*100:.1f}%",
        "p_desastre":        f"{probs[1]*100:.1f}%",
        "target":            pred,
    }


# --- Demostración con ejemplos ---
tweets_demo = [
    "BREAKING: Massive earthquake hits coastal city, thousands evacuated immediately",
    "Just had the best pizza of my life at this little place downtown 🍕😍",
    "Wildfire spreading rapidly near residential areas. Residents urged to evacuate NOW",
    "My heart is on fire every time I see you lol #love",
]

print("🔮 Demostración de Inferencia:\n")
for tweet in tweets_demo:
    resultado = predecir_tweet(tweet, modelo, tokenizador, DISPOSITIVO, umbral_opt)
    print(f"Tweet:      '{resultado['tweet']}'")
    print(f"Predicción: {resultado['prediccion']}")
    print(f"P(No Desastre): {resultado['p_no_desastre']} | P(Desastre): {resultado['p_desastre']}")
    print("-" * 70)

---
## 📝 Resumen y Recomendaciones para Mejorar la Puntuación

### ✅ Lo que hicimos:
- Fine-tuning de **DistilBERT** (`distilbert-base-uncased`) para clasificación binaria
- Limpieza de texto (URLs, menciones, caracteres repetidos)
- División estratificada 80/20 para entrenamiento/validación
- **AdamW** con decaimiento de learning rate lineal y warmup
- **Early stopping** para prevenir sobreajuste
- **Optimización del umbral** de decisión para maximizar F1

---

### 🚀 Estrategias para mejorar aún más el score:

1. **Usar `bert-base-uncased` o `roberta-base`**: Modelos más grandes con mayor capacidad. RoBERTa suele superar a BERT en tareas de clasificación de texto.

2. **Cross-Validation con Ensemble**: Entrenar con 5-fold CV y promediar las probabilidades de todos los modelos (bagging). Esto reduce la varianza considerablemente.

3. **Data Augmentation**: 
   - Back-translation (inglés → español → inglés)
   - Reemplazo de sinónimos
   - Mezcla de `keyword` + `location` + `text` como entrada

4. **Incorporar `keyword` y `location`**: Concatenar `[keyword] texto del tweet` puede aportar contexto valioso que el modelo solo con texto pierde.

5. **Pseudo-labeling**: Usar el modelo para etiquetar el test set con alta confianza y reentrenarlo (semi-supervisado).

6. **Ajustar Learning Rate más bajo (1e-5)**: Para modelos más grandes puede mejorar la convergencia.

7. **Longitud máxima mayor (160 tokens)**: Si hay tweets más largos con contexto importante.

---

### 📚 Referencias
- [DistilBERT Paper](https://arxiv.org/abs/1910.01108) - Sanh et al., 2019
- [HuggingFace Transformers](https://huggingface.co/docs/transformers/)
- [Fine-tuning BERT for Text Classification](https://arxiv.org/abs/1905.05583)
- [Competencia Kaggle NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started)

In [ ]:
# ============================================================
# Resumen final del experimento
# ============================================================

print("\n" + "="*60)
print("📋 RESUMEN FINAL DEL EXPERIMENTO")
print("="*60)
print(f"  Modelo base:          {CONFIG['modelo_base']}")
print(f"  Épocas entrenadas:    {len(historial['perdida_train'])}")
print(f"  Batch size:           {CONFIG['batch_size']}")
print(f"  Learning rate:        {CONFIG['learning_rate']}")
print(f"  Max longitud tokens:  {CONFIG['max_longitud']}")
print("-"*60)
print(f"  Mejor F1 validación:  {mejor_f1_val:.4f}")
print(f"  AUC-ROC:              {auc_score:.4f}")
print(f"  Accuracy:             {accuracy:.4f}")
print(f"  Umbral óptimo:        {umbral_opt:.2f}")
print("-"*60)
print(f"  Tweets en test:       {len(df_test):,}")
print(f"  Predicciones positivas: {preds_test.sum():,} ({preds_test.mean()*100:.1f}%)")
print("="*60)
print()
print("📁 Archivos generados:")
archivos_output = [
    ("/kaggle/working/submission.csv",              "Envío para Kaggle"),
    ("/kaggle/working/mejor_modelo.pt",             "Pesos del mejor modelo (PyTorch)"),
    ("/kaggle/working/eda_visualizaciones.png",     "Gráficos EDA"),
    ("/kaggle/working/curvas_entrenamiento.png",    "Curvas de entrenamiento"),
    ("/kaggle/working/evaluacion_modelo.png",       "Evaluación del modelo"),
    ("/kaggle/working/umbral_optimo.png",           "Optimización del umbral"),
    ("/kaggle/working/distilbert_tweets_desastre/", "Modelo HuggingFace completo"),
]
for ruta, descripcion in archivos_output:
    existe = os.path.exists(ruta)
    icono  = "✅" if existe else "❌"
    print(f"  {icono} {ruta}")
    print(f"     └─ {descripcion}")